# 00 — Shared functions

Everything that more than one notebook uses lives here, and only here. Other notebooks load it with:

```python
%run ./00_common.ipynb
```

Contents: configuration and keys · problem sets and the answer key · cost meter · model calls (Groq / OpenAI) ·
CV perception · CV verifier · panel montages · the one-call VLM perception · the grader · saving results.
Add a function here only when a second notebook needs it.

In [ ]:
import os, io, re, json, math, time, base64, random, zlib, itertools
from difflib import SequenceMatcher
from typing import Optional, TypedDict
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont
from scipy import ndimage as ndi
import cv2
import matplotlib.pyplot as plt

try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True))          # repo-root .env (copy .env.example)
except Exception:
    pass

_here = os.getcwd()
REPO = os.path.dirname(_here) if os.path.basename(_here) == "notebooks" else _here
DATA = os.path.join(REPO, "data", "bongard-classic")
PANELS = os.path.join(DATA, "panels")
RESULTS = os.path.join(REPO, "results")
for _sub in ("perception", "predictions", "grades", "tables", "figures"):
    os.makedirs(os.path.join(RESULTS, _sub), exist_ok=True)

VLM_MODEL = os.environ.get("VLM_MODEL", "gpt-4o")                   # the ONE vision call per problem
AGENT_MODEL = os.environ.get("AGENT_MODEL", "openai/gpt-oss-120b")  # small open reasoning agents (Groq)
GRADER_MODEL = os.environ.get("GRADER_MODEL", "openai/gpt-oss-120b")   # 20b misses reversed relations
HAVE_GROQ, HAVE_OPENAI = bool(os.environ.get("GROQ_API_KEY")), bool(os.environ.get("OPENAI_API_KEY"))

# chart colours, validated for colour-blind separation: blue = our system, orange = baselines
C_OURS, C_BASE, C_INK, C_INK2, C_SURFACE = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e", "#fcfcfb"
plt.rcParams.update({"figure.facecolor": C_SURFACE, "axes.facecolor": C_SURFACE, "axes.edgecolor": C_INK2,
                     "axes.labelcolor": C_INK, "xtick.color": C_INK2, "ytick.color": C_INK2, "text.color": C_INK,
                     "axes.spines.top": False, "axes.spines.right": False, "axes.grid": False, "font.size": 10})
print(f"repo: {REPO}")
print(f"keys: groq={HAVE_GROQ} openai={HAVE_OPENAI} | vision={VLM_MODEL} | agents={AGENT_MODEL} | grader={GRADER_MODEL}")

In [ ]:
SOLUTIONS = json.load(open(os.path.join(DATA, "solutions.json")))
PIDS = [f"{s}_{k}" for s in ("left", "right") for k in range(1, 7)]
ALL_100 = [f"bp{i:04d}" for i in range(1, 101) if os.path.isdir(os.path.join(PANELS, f"bp{i:04d}"))]
DEV_DESIGN = ['bp0002', 'bp0004', 'bp0007', 'bp0010', 'bp0031', 'bp0036', 'bp0047', 'bp0050', 'bp0056', 'bp0097', 'bp0049']       # used while designing the system
DEV_HELDOUT = ['bp0011', 'bp0022', 'bp0025', 'bp0028', 'bp0030', 'bp0033', 'bp0038', 'bp0046', 'bp0057', 'bp0096']      # never used for design decisions
DEV_HELDOUT2 = ['bp0001', 'bp0012', 'bp0016', 'bp0019', 'bp0020', 'bp0021', 'bp0042', 'bp0044', 'bp0054', 'bp0060', 'bp0072', 'bp0073', 'bp0076', 'bp0082', 'bp0084', 'bp0085', 'bp0086', 'bp0090', 'bp0091', 'bp0093']   # drawn once from outside DEV_21; judge design changes here
DEV_21 = DEV_DESIGN + DEV_HELDOUT
TEST_30 = DEV_HELDOUT + DEV_HELDOUT2        # every problem not used for design decisions
SMOKE = ["bp0002", "bp0036", "bp0038"]
PROBLEM_SETS = {"SMOKE": SMOKE, "DEV_21": DEV_21, "HELDOUT2": DEV_HELDOUT2, "TEST_30": TEST_30, "ALL_100": ALL_100}

def gt(bp):
    """Bongard's own answer (Foundalis solutions page)."""
    return SOLUTIONS[bp]["concept"]

def split_rule(rule):
    """'Left = X, Right = Y' -> (X, Y)."""
    m = re.match(r"\s*Left\s*=\s*(.*?)\s*,\s*Right\s*=\s*(.*)$", rule or "", re.I | re.S)
    if m:
        return m.group(1).strip().rstrip("."), m.group(2).strip().rstrip(".")
    return (rule or "").strip(), ""

def panel_path(bp, pid):
    return os.path.join(PANELS, bp, pid + ".png")

def save_json(obj, *parts):
    path = os.path.join(RESULTS, *parts)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=1, default=str)
    return path

def load_json(*parts, default=None):
    path = os.path.join(RESULTS, *parts)
    if not os.path.exists(path):
        return default
    with open(path) as f:
        return json.load(f)

print(len(ALL_100), "classic problems |", len(DEV_21), "development problems |", "sets:", list(PROBLEM_SETS))

In [ ]:
# USD per 1M tokens
PRICES = {
    "gpt-4o": {"in": 2.50, "out": 10.00}, "gpt-4o-mini": {"in": 0.15, "out": 0.60},
    "gpt-5": {"in": 1.25, "out": 10.00}, "gpt-5-mini": {"in": 0.25, "out": 2.00},
    "openai/gpt-oss-120b": {"in": 0.15, "out": 0.75}, "openai/gpt-oss-20b": {"in": 0.10, "out": 0.50},
}

class BudgetExceeded(Exception):
    pass

class Meter:
    """One spend ceiling per notebook run. An unpriced model is a hard error, never a silent $0."""
    def __init__(self, budget_usd):
        self.budget, self.total, self.calls, self.by_tag, self.tag = budget_usd, 0.0, 0, {}, "untagged"

    def record(self, model, tokens_in, tokens_out):
        if model not in PRICES:
            raise KeyError(f"add {model!r} to PRICES before using it")
        cost = tokens_in / 1e6 * PRICES[model]["in"] + tokens_out / 1e6 * PRICES[model]["out"]
        self.total += cost
        self.calls += 1
        self.by_tag[self.tag] = self.by_tag.get(self.tag, 0.0) + cost
        if self.total > self.budget:
            raise BudgetExceeded(f"spent ${self.total:.3f} > ceiling ${self.budget:.2f}")
        return cost

    def report(self):
        print(f"--- spend: ${self.total:.4f} of ${self.budget:.2f} ceiling, {self.calls} calls ---")
        for tag, cost in sorted(self.by_tag.items(), key=lambda kv: -kv[1]):
            print(f"   {tag:44s} ${cost:.4f}")

METER = Meter(budget_usd=float(os.environ.get("BUDGET_USD", "2.0")))

In [ ]:
_SYS = {"role": "system", "content": "You respond ONLY with a single valid JSON object and nothing else."}

def parse_json(text):
    """Parse a JSON reply; salvage the first {...} block if the model added prose."""
    t = (text or "").strip()
    if t.startswith("```"):
        t = t.split("```")[1]
        t = t[4:] if t.startswith("json") else t
    try:
        return json.loads(t)
    except Exception:
        m = re.search(r"\{.*\}", text or "", re.S)
        if m:
            return json.loads(m.group())
        raise

def _openai_call(model, messages, max_tokens, temperature):
    from openai import OpenAI
    kw = dict(model=model, messages=messages, response_format={"type": "json_object"})
    if model.startswith(("gpt-5", "o1", "o3", "o4")):        # reasoning models take no temperature
        kw["max_completion_tokens"] = max_tokens
    else:
        kw["max_tokens"], kw["temperature"] = max_tokens, temperature
    r = OpenAI(api_key=os.environ["OPENAI_API_KEY"]).chat.completions.create(**kw)
    return r.choices[0].message.content or "", r.usage.prompt_tokens, r.usage.completion_tokens

def _groq_call(model, messages, max_tokens, temperature, effort, json_mode=True):
    from groq import Groq
    kw = dict(model=model, messages=messages, max_tokens=max_tokens, temperature=temperature)
    if json_mode:
        kw["response_format"] = {"type": "json_object"}
    if "gpt-oss" in model:
        kw["reasoning_effort"] = effort
    r = Groq(api_key=os.environ["GROQ_API_KEY"]).chat.completions.create(**kw)
    return r.choices[0].message.content or "", r.usage.prompt_tokens, r.usage.completion_tokens

def _with_retry(fn, max_tokens):
    last = None
    for attempt in range(6):
        try:
            return fn(max_tokens)
        except BudgetExceeded:
            raise
        except Exception as e:                 # rate limit, empty or truncated JSON: wait, allow more room
            last = e
            max_tokens = int(max_tokens * 1.5)
            time.sleep(min(2 ** attempt, 20))
    raise last

def text_json(prompt, model=None, max_tokens=1500, temperature=0.0, effort="medium"):
    """One text call -> parsed JSON. gpt-oss models run on Groq, all others on OpenAI."""
    model = model or AGENT_MODEL
    msgs = [_SYS, {"role": "user", "content": prompt}]
    def once(mt):
        if model.startswith("openai/gpt-oss"):
            try:
                txt, ti, to = _groq_call(model, msgs, mt, temperature, effort)
            except Exception as e:           # Groq JSON mode sometimes returns an empty generation
                if "json_validate_failed" not in str(e):
                    raise
                txt, ti, to = _groq_call(model, msgs, mt, temperature, effort, json_mode=False)
        else:
            txt, ti, to = _openai_call(model, msgs, mt, temperature)
        METER.record(model, ti, to)
        return parse_json(txt)
    return _with_retry(once, max_tokens)

def vlm_json(prompt, image_png, model=None, max_tokens=800):
    """One vision call on one image -> parsed JSON."""
    model = model or VLM_MODEL
    b64 = base64.b64encode(image_png).decode()
    msgs = [_SYS, {"role": "user", "content": [
        {"type": "text", "text": prompt},
        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}", "detail": "high"}}]}]
    def once(mt):
        txt, ti, to = _openai_call(model, msgs, mt, 0.0)
        METER.record(model, ti, to)
        return parse_json(txt)
    return _with_retry(once, max_tokens)

## CV perception — each panel measured with OpenCV (free, deterministic)

In [ ]:
def _axes(ys, xs):
    """Major/minor-axis elongation and orientation from second moments (as skimage regionprops in the original)."""
    y, x = ys - ys.mean(), xs - xs.mean()
    cov = np.cov(np.vstack([x, y])) if len(xs) > 1 else np.eye(2)
    ev = np.sort(np.linalg.eigvalsh(cov))[::-1]
    major, minor = 4 * math.sqrt(max(ev[0], 0)), 4 * math.sqrt(max(ev[1], 0))
    return {"elongation": major / max(minor, 1e-6), "orientation": 0.5 * math.atan2(2 * cov[0, 1], cov[0, 0] - cov[1, 1])}

def _compactness(region):
    """4*pi*area / perimeter^2 (1.0 = a disc; low = spread out)."""
    cnts, _ = cv2.findContours(region.astype(np.uint8) * 255, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    perim = sum(cv2.arcLength(c, True) for c in cnts)
    return float(4 * math.pi * region.sum() / max(perim ** 2, 1e-6)) if cnts else 0.0

def _angles(region):
    """Corner angles of the figure's outline: any acute (25-80 deg) or right (80-100 deg) corner, whether it points
    out of the figure or into it. Turns sharper than 25 deg are the ends of thin lines, not corners; outlines with
    more than 12 corners are curves."""
    m = ndi.binary_fill_holes(region).astype(np.uint8) * 255
    cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not cnts:
        return {"acute": False, "right_angle": False}
    c = max(cnts, key=cv2.contourArea)
    poly = cv2.approxPolyDP(c, 0.02 * cv2.arcLength(c, True), True).reshape(-1, 2).astype(float)
    if not 3 <= len(poly) <= 12:
        return {"acute": False, "right_angle": False}
    acute = right = False
    for i in range(len(poly)):
        u, v = poly[i - 1] - poly[i], poly[(i + 1) % len(poly)] - poly[i]
        ang = math.degrees(math.acos(np.clip(u @ v / max(np.linalg.norm(u) * np.linalg.norm(v), 1e-6), -1, 1)))
        acute |= 25 <= ang < 80
        right |= 80 <= ang <= 100
    return {"acute": bool(acute), "right_angle": bool(right)}

INK_THRESHOLD = 128      # gray < this = ink
MIN_OBJECT_AREA = 80     # drop specks smaller than this
SOLID_FILL = 0.75        # fill_ratio above this = a solid figure  (the debuggable knob)
OUTLINE_FILL = 0.45      # fill_ratio below this = an outline figure

def _approx_ncorners(region):
    """From: services/cv_perception.py::_approx_ncorners. cv2 polygon-approx corner count."""
    m = region.astype(np.uint8) * 255
    cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts: return 0
    c = max(cnts, key=cv2.contourArea); peri = cv2.arcLength(c, True)
    if peri < 1e-6 or len(c) < 4: return 0
    return int(len(cv2.approxPolyDP(c, 0.015 * peri, True)))     # epsilon = 1.5% of perimeter

def _silhouette_convexity(filled):
    """From: services/cv_perception.py::_silhouette_convexity. filled area / hull area (1.0=convex)."""
    m = filled.astype(np.uint8) * 255
    cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts: return 1.0
    c = max(cnts, key=cv2.contourArea)
    area, harea = cv2.contourArea(c), cv2.contourArea(cv2.convexHull(c))
    return float(min(1.0, area / harea)) if harea > 1e-6 else 1.0

def _classify_shape(ncorners, elong):
    """From: services/cv_perception.py::_classify_shape."""
    if ncorners <= 0: return "other"
    if ncorners == 3: return "triangle"
    if 4 <= ncorners <= 6: return "rectangle"
    if ncorners >= 7: return "circle"
    return "other"

def _is_panel_frame(bbox, fill_ratio, ph, pw):
    """From: services/cv_perception.py::_is_panel_frame. Drop the rectangular border."""
    minc, minr, maxc, maxr = bbox
    return (maxc-minc) >= 0.92*pw and (maxr-minr) >= 0.92*ph and fill_ratio < 0.25

def _measure_object(mask_label, ph, pw):
    """From: services/cv_perception.py::_measure_object. Size=bbox extent; fill via binary_fill_holes."""
    ys, xs = np.where(mask_label)
    if len(xs) < MIN_OBJECT_AREA: return None
    minr, maxr, minc, maxc = ys.min(), ys.max()+1, xs.min(), xs.max()+1
    region = mask_label[minr:maxr, minc:maxc]
    area_frac = max(1, region.size) / float(ph*pw)
    filled = ndi.binary_fill_holes(region)
    enclosed = int(filled.sum()) or int(region.sum())
    fill_ratio = float(region.sum()) / max(enclosed, 1)          # ink / enclosed area
    cy, cx = ys.mean(), xs.mean()
    h, w = (maxr-minr), (maxc-minc)
    return dict(area=int(region.sum()), area_frac=area_frac,
                centroid_x=cx/max(pw-1,1), centroid_y=cy/max(ph-1,1),
                convexity=_silhouette_convexity(filled),
                **_axes(ys, xs), compactness=_compactness(region), **_angles(region),
                ncorners=_approx_ncorners(region), fill_ratio=fill_ratio,
                is_solid=fill_ratio > SOLID_FILL, is_outline=(fill_ratio < OUTLINE_FILL and region.sum() > MIN_OBJECT_AREA),
                shape_class=_classify_shape(_approx_ncorners(region), 0),
                bbox=[int(minc), int(minr), int(maxc), int(maxr)])

def _panel_topology(objs):
    """From: services/cv_perception.py::_panel_topology. Box-level inside / overlap."""
    boxes = [o["bbox"] for o in objs]
    def contains(a, b):
        aa=(a[2]-a[0])*(a[3]-a[1]); ab=(b[2]-b[0])*(b[3]-b[1])
        return a[0]<=b[0] and a[1]<=b[1] and a[2]>=b[2] and a[3]>=b[3] and aa>ab
    inside = any(contains(boxes[i], boxes[j]) for i in range(len(boxes)) for j in range(len(boxes)) if i!=j)
    overlap = False
    for i in range(len(boxes)):
        for j in range(i+1, len(boxes)):
            a, b = boxes[i], boxes[j]
            if not (a[2]<=b[0] or b[2]<=a[0] or a[3]<=b[1] or b[3]<=a[1]) and not contains(a,b) and not contains(b,a):
                overlap = True
    return {"any_inside": inside, "any_overlap": overlap}

def _panel_symmetric(ink, thresh=0.90):
    """From: services/cv_perception.py::_panel_symmetric. Mirror agreement L-R or T-B."""
    return float((ink==np.fliplr(ink)).mean()) >= thresh or float((ink==np.flipud(ink)).mean()) >= thresh

def extract_cv_features(image_bytes, side):
    """From: services/cv_perception.py::extract_cv_features. One panel -> features + side_stats."""
    gray = np.array(Image.open(io.BytesIO(image_bytes)).convert("L"), dtype=np.float32)
    ph, pw = gray.shape
    ink = ndi.binary_closing(gray < INK_THRESHOLD, iterations=1)
    labeled, n = ndi.label(ink)
    objs = []
    for lab in range(1, n+1):
        o = _measure_object(labeled == lab, ph, pw)
        if o and not _is_panel_frame(o["bbox"], o["fill_ratio"], ph, pw):
            objs.append(o)
    topo = _panel_topology(objs); areas = [o["area_frac"] for o in objs]
    side_stats = {
        "mean_area_frac":  float(np.mean(areas)) if areas else 0.0,
        "mean_fill_ratio": float(np.mean([o["fill_ratio"] for o in objs])) if objs else 0.0,
        "mean_convexity":  float(np.mean([o["convexity"] for o in objs])) if objs else 0.0,
        "all_solid":   all(o["is_solid"] for o in objs) if objs else False,
        "all_outline": all(o["is_outline"] for o in objs) if objs else False,
        "shape_classes": list({o["shape_class"] for o in objs}),
        "all_curved":   bool(objs) and all(o["shape_class"]=="circle" for o in objs),
        "all_straight": bool(objs) and all(o["shape_class"] in ("triangle","rectangle") for o in objs),
        "any_inside": topo["any_inside"], "any_overlap": topo["any_overlap"],
        "symmetric": _panel_symmetric(ink),
    }
    return {"side": side, "object_count": len(objs), "objects": objs, "side_stats": side_stats}

## CV verifier and concept language — does a rule separate all six left panels from all six right ones?

Objective, needs no answer key. Faithful to the original `services/concept_language.py` (predicates, priorities,
side-aware thresholds, contrast-ranked search, optional conjunction search), with two fixes:
categorical rules are now tested in **both** directions (the original only ever tested "holds on the left"), and the
check of free-text rules (`verify_text_rule`) respects **which side** a property is claimed for.

In [ ]:
NUMERIC = ("fill_ratio", "area_frac", "convexity", "compactness", "elongation", "ncorners",
           "centroid_x", "centroid_y", "object_count", "aspect")
CATEGORICAL = ("outline", "solid", "curved", "straight", "inside", "overlap", "symmetric",
               "triangle", "rectangle", "circle", "large", "small")
PRIORITY = {"fill_ratio": 100, "outline": 95, "solid": 95, "object_count": 92, "area_frac": 90, "large": 85,
            "small": 85, "triangle": 80, "rectangle": 80, "circle": 80, "aspect": 75, "centroid_x": 70,
            "centroid_y": 70, "inside": 68, "overlap": 66, "curved": 64, "straight": 62, "symmetric": 60,
            "compactness": 40, "convexity": 30, "elongation": 25, "ncorners": 20}
NL = {  # (label when the property is LOW / present on the left, label for the other side)
    "solid": ("solid figures", "outline figures"), "outline": ("outline figures", "solid figures"),
    "triangle": ("triangles", "non-triangles"), "rectangle": ("rectangles", "non-rectangles"),
    "circle": ("circles", "non-circles"), "curved": ("curved figures", "straight-edged figures"),
    "straight": ("straight-edged figures", "curved figures"),
    "inside": ("one figure inside another", "figures kept separate"),
    "overlap": ("overlapping figures", "figures kept separate"),
    "symmetric": ("symmetric figures", "asymmetric figures"),
    "large": ("large figures", "small figures"), "small": ("small figures", "large figures"),
    "fill_ratio": ("outline figures", "solid figures"), "area_frac": ("small figures", "large figures"),
    "convexity": ("nonconvex figures", "convex figures"),
    "centroid_x": ("figures on the left side", "figures on the right side"),
    "centroid_y": ("figures at the top", "figures at the bottom"),
    "object_count": ("fewer figures", "more figures"),
    "aspect": ("figures elongated vertically", "figures elongated horizontally"),
    "compactness": ("irregular, spread-out figures", "compact figures"),
    "elongation": ("compact figures", "elongated figures"),
    "ncorners": ("figures with fewer corners", "figures with more corners"),
}

def _panel_stat(feat, attr):
    """Per-panel value of a numeric attribute (mean over the panel's objects)."""
    if attr == "object_count":
        return float(feat["object_count"])
    objs = feat["objects"]
    if not objs:
        return 0.0
    if attr == "aspect":
        return float(np.mean([max(o["bbox"][2] - o["bbox"][0], 1) / max(o["bbox"][3] - o["bbox"][1], 1) for o in objs]))
    return float(np.mean([o.get(attr, 0.0) for o in objs]))

def _holds(pred, feat):
    """Does the categorical property hold on this panel?"""
    ss, objs = feat["side_stats"], feat["objects"]
    if pred == "solid":
        return ss["all_solid"] or ss["mean_fill_ratio"] > SOLID_FILL
    if pred == "outline":
        return ss["all_outline"] or ss["mean_fill_ratio"] < OUTLINE_FILL
    if pred in ("triangle", "rectangle", "circle"):
        return pred in ss["shape_classes"]
    if pred == "large":
        return bool(objs) and max(o["area_frac"] for o in objs) > 0.05
    if pred == "small":
        return bool(objs) and max(o["area_frac"] for o in objs) < 0.04
    return bool({"curved": ss["all_curved"], "straight": ss["all_straight"], "inside": ss["any_inside"],
                 "overlap": ss["any_overlap"], "symmetric": ss["symmetric"]}.get(pred, False))

def _eval_predicate(pred, left_holds, threshold, feat):
    """Does this panel satisfy the LEFT side of the rule?"""
    if pred in NUMERIC:
        v = _panel_stat(feat, pred)
        return (v < threshold) if left_holds else (v >= threshold)
    return _holds(pred, feat) if left_holds else not _holds(pred, feat)   # fixed: the direction is applied

def verify_rule(pred, left_holds, threshold, pool):
    """All six left panels satisfy the left side and no right panel does."""
    fl = [i for i, f in pool.items() if f["side"] == "left" and not _eval_predicate(pred, left_holds, threshold, f)]
    fr = [i for i, f in pool.items() if f["side"] == "right" and _eval_predicate(pred, left_holds, threshold, f)]
    return (not fl and not fr), fl, fr

def _numeric_threshold(pred, pool):
    """Side-aware threshold: midpoint between the two clusters; left_holds = left is the LOW side."""
    L = [_panel_stat(f, pred) for f in pool.values() if f["side"] == "left"]
    R = [_panel_stat(f, pred) for f in pool.values() if f["side"] == "right"]
    if not L or not R:
        return None, None
    return ((max(L) + min(R)) / 2.0, True) if np.mean(L) < np.mean(R) else ((min(L) + max(R)) / 2.0, False)

def rule_nl(pred, left_holds):
    la, ra = NL.get(pred, (pred, "not " + pred))
    return f"Left = {la}, Right = {ra}" if left_holds else f"Left = {ra}, Right = {la}"

def _trials(pool, allowed=None):
    out = []
    for p in NUMERIC:
        if allowed is None or p in allowed:
            thr, lh = _numeric_threshold(p, pool)
            if thr is not None:
                out.append((p, lh, thr))
    for p in CATEGORICAL:
        if allowed is None or p in allowed:
            out += [(p, True, None), (p, False, None)]
    return out

def _verify_conjunction(a, b, pool):
    """AND of two rules: a panel is on the left iff both rules put it there."""
    fl = [i for i, f in pool.items() if f["side"] == "left" and not (_eval_predicate(*a, f) and _eval_predicate(*b, f))]
    fr = [i for i, f in pool.items() if f["side"] == "right" and (_eval_predicate(*a, f) and _eval_predicate(*b, f))]
    return not fl and not fr

def propose_verified_rule(pool, contrast=None, allow_conjunction=False):
    """Best single rule that separates the panels, ranked by priority + contrast score (Occam: singles first).
    With allow_conjunction, fall back to the AND of two rules (over-fit-prone; off in the pipeline)."""
    scores = {c["attribute"]: c["score"] for c in (contrast or {}).get("candidates", [])}
    best = None
    for pred, lh, thr in _trials(pool):
        if verify_rule(pred, lh, thr, pool)[0]:
            rank = PRIORITY.get(pred, 10) + scores.get(pred, 0.0)
            if best is None or rank > best[0]:
                best = (rank, pred, lh, thr)
    if best:
        _, pred, lh, thr = best
        return {"predicate": pred, "left_holds": lh, "threshold": thr, "nl": rule_nl(pred, lh)}
    if allow_conjunction:
        trials = sorted(_trials(pool), key=lambda t: -PRIORITY.get(t[0], 10))
        for i in range(len(trials)):
            for j in range(i + 1, len(trials)):
                a, b = trials[i], trials[j]
                if a[0] != b[0] and _verify_conjunction(a, b, pool):
                    la = rule_nl(a[0], a[1]).split(",")[0].replace("Left = ", "")
                    lb = rule_nl(b[0], b[1]).split(",")[0].replace("Left = ", "")
                    return {"predicate": f"{a[0]}&{b[0]}", "left_holds": True, "threshold": None,
                            "nl": f"Left = {la} and {lb}, Right = not both"}
    return None

def verified_predicates(pool, allowed=None):
    """Every single predicate that, on its own, separates all 12 panels (value function for feature Shapley)."""
    return {p for p, lh, thr in _trials(pool, allowed) if verify_rule(p, lh, thr, pool)[0]}

# Checking a free-text rule: keywords -> predicate + the direction the text claims.
_KW = [  # (predicate, words for HIGH / present, words for LOW / absent)   numeric: HIGH = large value
    ("area_frac", ["large", "big", "bigger", "larger"], ["small", "tiny", "smaller", "little"]),
    ("object_count", ["more figures", "many", "multiple", "two ", "three ", "several"], ["fewer", "one figure", "single figure", "few "]),
    ("aspect", ["horizontal", "wide", "horizontally"], ["vertical", "tall", "vertically"]),
    ("elongation", ["elongated", "stretched", "long "], ["compact"]),
    ("convexity", ["convex"], ["concave"]),                        # "nonconvex" = negated "convex"
    ("centroid_x", ["on the right", "right side"], ["on the left", "left side"]),
    ("centroid_y", ["bottom", "lower part"], ["top", "upper part"]),
    ("outline", ["outline", "hollow", "unfilled", "empty"], []),
    ("solid", ["solid", "filled", "black"], []),
    ("triangle", ["triangle"], []), ("circle", ["circle", "circular"], []),
    ("rectangle", ["rectangle", "square", "quadrangle", "quadrilateral"], []),
    ("curved", ["curved", "curvilinear"], []), ("straight", ["straight", "polygon"], []),
    ("inside", ["inside", "contain", "within", "enclosed"], []), ("overlap", ["overlap", "intersect"], []),
    ("symmetric", ["symmetric", "symmetry"], []),
]
def _mentions(text, words):
    """'pos' if a word appears plainly, 'neg' if right after no/not/non-/without, else None."""
    t = f" {text.lower()} "
    for w in words:
        i = t.find(w)
        if i >= 0:
            return "neg" if re.search(r"(\bno|\bnot|\bnon-?|\bwithout)\s*$", t[max(0, i - 10):i]) else "pos"
    return None

def verify_text_rule(rule, pool):
    """Does a free-text 'Left = X, Right = Y' rule verify against the CV measurements, in the direction it claims?
    Returns the verified predicate or None. Most VLM-level concepts are not CV-expressible and return None."""
    L, R = split_rule(rule)
    if not L:
        return None
    for pred, hi, lo in _KW:
        if pred in NUMERIC:
            lh_, ll_, rh_, rl_ = _mentions(L, hi), _mentions(L, lo), _mentions(R, hi), _mentions(R, lo)
            left_high = "pos" in (lh_, rl_) or "neg" in (ll_, rh_)
            left_low = "pos" in (ll_, rh_) or "neg" in (lh_, rl_)
            if left_high == left_low:                  # not mentioned, or contradictory
                continue
            thr, lh = _numeric_threshold(pred, pool)
            if thr is None or lh != left_low:          # the data's direction must match the claimed one
                continue
            if verify_rule(pred, lh, thr, pool)[0]:
                return pred
        else:
            ml, mr = _mentions(L, hi), _mentions(R, hi)
            claims = []
            if ml == "pos" or mr == "neg":
                claims.append(True)
            if mr == "pos" or ml == "neg":
                claims.append(False)
            for lh in claims:
                if verify_rule(pred, lh, None, pool)[0]:
                    return pred
    return None

## Contrast — rank measured attributes by how well they split left from right (original `services/contrast.py`)

In [ ]:
CONTRAST_NUMERIC = ("area_frac", "fill_ratio", "convexity", "compactness", "elongation", "ncorners",
                    "centroid_x", "centroid_y", "object_count", "aspect")
CONTRAST_CATEGORICAL = ("solid", "outline", "triangle", "rectangle", "circle", "large", "small")

def _panel_values(pool, side, attr):
    vals = []
    for f in pool.values():
        if f["side"] != side or not f["objects"]:
            continue
        ss, objs = f["side_stats"], f["objects"]
        if attr in ("solid", "outline"):
            vals.append(1.0 if ss["all_" + attr] else 0.0)
        elif attr in ("triangle", "rectangle", "circle"):
            vals.append(1.0 if attr in ss["shape_classes"] else 0.0)
        elif attr == "large":
            vals.append(max(o["area_frac"] for o in objs))
        elif attr == "small":
            vals.append(min(o["area_frac"] for o in objs))
        else:
            vals.append(_panel_stat(f, attr))
    return vals

def _uniformity(v):
    if len(v) <= 1:
        return 1.0
    m, sd = float(np.mean(v)), float(np.std(v))
    if abs(m) < 1e-9:
        return 1.0 if sd < 1e-9 else 0.0
    return max(0.0, 1.0 - min(sd / abs(m), 1.0))

def compute_contrast(pool, attrs=None):
    """Score = effect size x uniformity(left) x uniformity(right); keep effect > 0.5, top 10."""
    cands = []
    for a in attrs or (CONTRAST_NUMERIC + CONTRAST_CATEGORICAL):
        L, R = _panel_values(pool, "left", a), _panel_values(pool, "right", a)
        if not L or not R:
            continue
        effect = abs(np.mean(L) - np.mean(R)) / max(float(np.std(L + R)), 1e-6)
        if effect <= 0.5:
            continue
        overlap, gap = _separation(L, R)
        cands.append({"attribute": a, "effect": float(effect), "left_mean": float(np.mean(L)), "right_mean": float(np.mean(R)),
                      "score": float(effect * _uniformity(L) * _uniformity(R)), "overlap": overlap, "gap": gap})
    cands.sort(key=lambda c: -c["score"])
    return {"candidates": cands[:10], "top": cands[0]["attribute"] if cands else None}

def _separation(L, R):
    """Panels on the wrong side of the best single cut, and the difference between the side means relative to
    the larger mean. The original effect size ignores both, so 0.989 vs 0.988 could rank as a strong clue."""
    vals = sorted(set(L + R))
    cuts = [(a + b) / 2 for a, b in zip(vals, vals[1:])] or vals
    wrong = min(min(sum(x <= t for x in L) + sum(x > t for x in R), sum(x > t for x in L) + sum(x <= t for x in R))
                for t in cuts)
    gap = abs(np.mean(L) - np.mean(R)) / max(abs(np.mean(L)), abs(np.mean(R)), 1e-9)
    return int(wrong), float(gap)

SEP_MAX_OVERLAP, SEP_MIN_GAP = 1, 0.15     # a clue must split the sides with at most one exception, by >= 15%
CONTRAST_WORDS = {   # (words when the value is HIGH, words when it is LOW)
    "area_frac": ("larger figures", "smaller figures"), "large": ("larger figures", "smaller figures"),
    "small": ("larger figures", "smaller figures"), "fill_ratio": ("filled figures", "outline figures"),
    "convexity": ("convex figures", "figures with dents"), "compactness": ("compact, rounded figures", "irregular, spread-out figures"),
    "elongation": ("elongated figures", "compact figures"), "ncorners": ("figures with more corners", "figures with fewer corners"),
    "centroid_x": ("figures toward the right", "figures toward the left"),
    "centroid_y": ("figures toward the bottom", "figures toward the top"),
    "object_count": ("more separate figures", "fewer separate figures"),
    "aspect": ("figures wider than tall", "figures taller than wide"),
    "solid": ("solid figures", "figures that are not solid"), "outline": ("outline figures", "figures that are not outlines"),
    "triangle": ("a triangle", "no triangle"), "rectangle": ("a four-sided figure", "no four-sided figure"),
    "circle": ("a round figure", "no round figure"),
}

def contrast_summary(contrast, attrs=None, k=3):
    """The contrast in words, keeping only attributes that really separate the sides; no numbers."""
    lines, seen = [], set()
    for c in contrast["candidates"]:
        if (attrs is not None and c["attribute"] not in attrs) or c.get("overlap", 99) > SEP_MAX_OVERLAP \
                or c.get("gap", 0) < SEP_MIN_GAP:
            continue
        hi, lo = CONTRAST_WORDS.get(c["attribute"], (f"higher {c['attribute']}", f"lower {c['attribute']}"))
        left, right = (hi, lo) if c["left_mean"] > c["right_mean"] else (lo, hi)
        if (left, right) in seen:
            continue
        seen.add((left, right))
        lines.append(f"  - left: {left}; right: {right}" + (" (one panel is an exception)" if c["overlap"] else ""))
        if len(lines) == k:
            break
    if not lines:
        return "MEASURED DIFFERENCES: no measured attribute separates the two sides."
    return "MEASURED DIFFERENCES between the sides (exact measurement):\n" + "\n".join(lines)

CV_MIN_MARGIN = 0.15
def cv_rule_margin(rule, pool):
    """How clearly a verified CV rule splits the sides: for a measured attribute, the gap between the closest left
    and right panels as a share of the attribute's full range (near 0 = a lucky split); categorical rules are exact."""
    pred = rule["predicate"]
    if pred not in NUMERIC:
        return 1.0
    L = [_panel_stat(f, pred) for f in pool.values() if f["side"] == "left"]
    R = [_panel_stat(f, pred) for f in pool.values() if f["side"] == "right"]
    gap = (min(R) - max(L)) if rule["left_holds"] else (min(L) - max(R))
    span = max(L + R) - min(L + R)
    return max(0.0, gap) / span if span > 1e-9 else 0.0

def contrast_text(contrast, attrs=None):
    """The contrast ranking as a prompt block (optionally only some attributes)."""
    cs = [c for c in contrast["candidates"] if attrs is None or c["attribute"] in attrs][:5]
    if not cs:
        return "RULER CONTRAST: no strong measured discriminator."
    lines = [f"  {c['attribute']}: {'left > right' if c['left_mean'] > c['right_mean'] else 'right > left'} "
             f"(effect={c['effect']:.2f}, score={c['score']:.2f}, L={c['left_mean']:.3f} R={c['right_mean']:.3f})" for c in cs]
    return "RULER CONTRAST (measured attributes ranked by how well they split the sides):\n" + "\n".join(lines)

def best_contrast_nl(pool, contrast):
    """Unverified best guess from the top discriminator (the contrast baseline's fallback)."""
    for c in contrast["candidates"]:
        a = c["attribute"]
        if a in NUMERIC:
            thr, lh = _numeric_threshold(a, pool)
            if thr is not None:
                return rule_nl(a, lh)
        else:
            return rule_nl(a, c["left_mean"] > c["right_mean"])
    return ""

## Panels and montages

In [ ]:
_CV = {}
def load_cv(bp):
    """CV measurements for the 12 panels of a problem (cached in memory)."""
    if bp not in _CV:
        _CV[bp] = {pid: extract_cv_features(open(panel_path(bp, pid), "rb").read(), pid.split("_")[0])
                   for pid in PIDS if os.path.exists(panel_path(bp, pid))}
    return _CV[bp]

def _font(size):
    try:
        return ImageFont.load_default(size=size)
    except Exception:
        return ImageFont.load_default()

def _panel(bp, pid, sz):
    return Image.open(panel_path(bp, pid)).convert("L").resize((sz, sz))

def std_montage(bp, sz=200, pad=10, gap=40):
    """The classic presentation: top row = the 6 left panels, bottom row = the 6 right panels."""
    W, H = 6 * (sz + pad) + pad, 2 * sz + 2 * pad + gap
    img = Image.new("L", (W, H), 255)
    d = ImageDraw.Draw(img)
    for r, side in enumerate(("left", "right")):
        for k in range(6):
            img.paste(_panel(bp, f"{side}_{k+1}", sz), (pad + k * (sz + pad), pad + r * (sz + gap)))
    d.line([(pad, pad + sz + gap // 2), (W - pad, pad + sz + gap // 2)], fill=0, width=2)
    buf = io.BytesIO(); img.save(buf, format="PNG")
    return buf.getvalue(), img

def shuffled_grid(bp, seed, sz=240, pad=10, label_h=28):
    """All 12 panels shuffled into a 3x4 grid labelled P1..P12, so the viewer cannot tell left from right."""
    order = PIDS[:]
    random.Random(seed).shuffle(order)
    W, H = 4 * (sz + pad) + pad, 3 * (sz + label_h + pad) + pad
    img = Image.new("L", (W, H), 255)
    d, f = ImageDraw.Draw(img), _font(22)
    labels = {}
    for i, pid in enumerate(order):
        r, c = divmod(i, 4)
        x, y = pad + c * (sz + pad), pad + r * (sz + label_h + pad)
        d.text((x + 4, y + 2), f"P{i+1}", fill=0, font=f)
        img.paste(_panel(bp, pid, sz), (x, y + label_h))
        labels[f"P{i+1}"] = pid
    buf = io.BytesIO(); img.save(buf, format="PNG")
    return buf.getvalue(), labels, img

def show_problem(bp, title=None):
    _, img = std_montage(bp)
    plt.figure(figsize=(11, 4)); plt.imshow(img, cmap="gray"); plt.axis("off")
    plt.title(title or f"{bp}: top row = LEFT, bottom row = RIGHT   (answer: {gt(bp)})")
    plt.show()

## Perception — divided labour: CV is the ruler, the VLM is the eye (used by 01, 02, 03)

Every fact the agents read has exactly one source, so the two never contradict each other:

| Source | Owns |
|---|---|
| **CV — ruler** (free, exact) | size, position, orientation (long-axis direction), elongation, convexity, corners, **corner angles** (acute / right), blob count, box-level inside / overlap |
| **VLM — eye** (one call per problem) | object type, **number of sides**, fill, pointing, count, relations, lines and self-crossing, symmetry, texture |

CV symmetry, shape class and fill are *not* shown to the agents: an audit of 252 panels found CV
"symmetric" on 92% of panels and shape class disagreeing with the VLM on 41%. The CV blob count is shown only
to the quantity specialist, labelled "touching shapes count as one" (it was off on 28% of panels).
The eye no longer reports size, position, orientation or angles: the ruler measures those exactly, and the eye
missed acute angles on half the panels of bp0033.

The eye writes a compact reply (short keys, defaults left out) because its output is most of the pipeline's
cost. `FIELDS_VERSION=fields-v2` in `.env` reuses the older cached readings; their size, position,
orientation and angle entries are simply not shown.

In [ ]:
FIELDS_PROMPT = """The image shows 12 small, independent picture panels labelled P1 to P12.
Look at each panel carefully and describe EVERY panel with the same short keys.
Do NOT look for any rule or grouping between panels. Leave out any key whose value is the default.

Keys for each panel:
  o    : list of objects, each {"t": type, "s": sides, "f": "solid", "p": pointing}
         t = triangle|circle|square|rectangle|ellipse|polygon|line|curve|dot|cross|arrow|other
         s = number of straight sides (3, 4, 5, 6, or "many"); leave out for circles, ellipses, lines, curves, dots
         f = "solid" if filled in (leave out for outline figures)
         p = up|down|left|right if the object clearly points that way (leave out otherwise)
  n    : number of separate objects
  ln   : number of separate lines or curves (leave out if it equals n)
  open : true if some line or curve is open, not a closed loop (default false)
  x    : true if a single line or curve crosses itself (default false)
  rel  : short phrases between objects, e.g. "triangle inside circle", "triangle above circle",
         "triangle larger than circle", "touching", "overlapping", "identical copies",
         "same shape, different size" (leave out if only one object)
  sym  : true if the panel's figure is mirror-symmetric, else false (always give)
  tex  : hatched-horizontal|hatched-vertical|hatched-diagonal|dotted (leave out if none)
  note : at most 10 words, only if something important is not covered above

Example: {"P1": {"o": [{"t": "triangle", "s": 3, "f": "solid", "p": "up"}, {"t": "circle"}], "n": 2,
          "rel": ["triangle inside circle"]}, "P2": {...}, ..., "P12": {...}}
Reply json with all 12 panels."""
FIELDS_VERSION = os.environ.get("FIELDS_VERSION", "fields-v3")

def _expand_fields(p):
    """A compact fields-v3 reading -> the full field names used everywhere else (older readings pass through)."""
    if not isinstance(p, dict) or "o" not in p and "n" not in p:
        return p if isinstance(p, dict) else {}
    objs = [{"type": o.get("t"), "sides": o.get("s"), "fill": o.get("f") or "outline", "pointing": o.get("p")}
            for o in p.get("o") or [] if isinstance(o, dict)]
    n = p.get("n", len(objs))
    return {"objects": objs, "count": n,
            "lines": {"separate_lines": p.get("ln", n), "closed": not p.get("open", False), "self_crossing": bool(p.get("x", False))},
            "relations": p.get("rel") or [], "symmetric": p.get("sym"), "texture": p.get("tex"), "notes": p.get("note")}

def vlm_fields(bp, model=None):
    """ONE vision call per problem: the eye's fields for all 12 panels. Cached on disk with its cost."""
    model = model or VLM_MODEL
    cache = ("perception", f"{bp}__{model}__{FIELDS_VERSION}.json")
    hit = load_json(*cache)
    if not hit:
        png, labels, _ = shuffled_grid(bp, seed=zlib.crc32(f"{bp}-fields".encode()))
        before = METER.total
        out = vlm_json(FIELDS_PROMPT, png, model=model, max_tokens=3000)
        hit = {"bp": bp, "model": model, "cost_usd": METER.total - before,
               "fields": {pid: out.get(lab, {}) for lab, pid in labels.items()}}
        save_json(hit, *cache)
    return {pid: _expand_fields(f) for pid, f in hit["fields"].items()}

def perception_cost(bp, model=None):
    hit = load_json("perception", f"{bp}__{model or VLM_MODEL}__{FIELDS_VERSION}.json")
    return hit["cost_usd"] if hit else 0.0

EYE_OBJ = ("fill", "type", "sides", "pointing")
EYE_KEYS = ("count", "lines", "relations", "symmetric", "texture", "notes")
RULER_KEYS = ("size", "position", "elongation", "convexity", "corners", "angles")

def eye_text(f, obj=EYE_OBJ, keys=EYE_KEYS):
    """The eye's reading of one panel, restricted to the requested object attributes and fields."""
    if not isinstance(f, dict) or not f:
        return "(no reading)"
    objs = []
    for o in f.get("objects") or []:
        if not isinstance(o, dict):
            continue
        words = [str(o.get(k)) for k in ("fill", "type") if k in obj and o.get(k) and o.get(k) != "none"]
        s = " ".join(words) or "object"
        if "sides" in obj and o.get("sides") not in (None, "", "none"):
            s += f" ({o['sides']} sides)"
        if "pointing" in obj and o.get("pointing") not in (None, "", "none"):
            s += f", pointing {o['pointing']}"
        objs.append(s)
    parts = [f"objects: {'; '.join(objs) or 'none'}"]
    ln = f.get("lines") or {}
    if "count" in keys:
        parts.append(f"count {f.get('count', '?')}")
    if "lines" in keys:
        parts.append(f"lines {ln.get('separate_lines', '?')}, {'closed' if ln.get('closed') else 'open'}, "
                     f"{'self-crossing' if ln.get('self_crossing') else 'no self-crossing'}")
    if "relations" in keys:
        parts.append("relations: " + ("; ".join(map(str, f.get("relations") or [])) or "none"))
    if "symmetric" in keys and f.get("symmetric") is not None:
        parts.append("symmetric" if f.get("symmetric") else "not symmetric")
    if "texture" in keys and f.get("texture") not in (None, "", "none"):
        parts.append(f"texture {f['texture']}")
    if "notes" in keys and f.get("notes"):
        parts.append(f"notes: {f['notes']}")
    return " | ".join(parts)

def _long_axis(o):
    minc, minr, maxc, maxr = o["bbox"]
    w, h = max(maxc - minc, 1), max(maxr - minr, 1)
    return "vertical" if h > 1.2 * w else "horizontal" if w > 1.2 * h else "none"

def _angle_words(o):
    flags = [w for k, w in (("acute", "acute corner"), ("right_angle", "right-angle corner")) if o.get(k)]
    return ", ".join(flags) or "no sharp corners"

def _form(o):
    """A figure named by its measured outline, so relations can say WHICH figure (e.g. '3-corner', 'round')."""
    n = o["ncorners"]
    if n >= 7 and not o.get("acute") and not o.get("right_angle"):
        return "round"
    return f"{n}-corner" if n else "irregular"

def _relative(a, b):
    """Where figure a sits relative to figure b, on both axes when both are clear."""
    dx, dy = a["centroid_x"] - b["centroid_x"], a["centroid_y"] - b["centroid_y"]
    parts = ([("above" if dy < 0 else "below")] if abs(dy) > 0.08 else []) + \
            ([("left of" if dx < 0 else "right of")] if abs(dx) > 0.08 else [])
    return " and ".join(parts) or "at the same place as"

def _rank_text(objs):
    """The two largest figures compared: which is bigger, and where it sits relative to the other."""
    if len(objs) < 2:
        return "one figure"
    a, b = objs[0], objs[1]
    size = "clearly bigger than" if a["area_frac"] > 1.5 * b["area_frac"] else "about the same size as"
    fa, fb = _form(a), _form(b)
    if fa == fb:
        fa, fb = "larger " + fa, "smaller " + fb
    return f"the {fa} figure is {size} the {fb} figure and sits {_relative(a, b)} it"

def ruler_text(feat, keys=RULER_KEYS):
    """The ruler's exact measurements of one panel's figures (touching shapes merge into one figure)."""
    objs = sorted(feat["objects"], key=lambda o: -o["area_frac"])
    head = []
    if "count" in keys:
        head.append(f"measured figure count {feat['object_count']} (touching shapes count as one)")
    if "topology" in keys:
        ss = feat["side_stats"]
        head.append("boxes: " + ("one inside another" if ss["any_inside"] else "overlapping" if ss["any_overlap"] else "separate"))
    if "rank" in keys:
        head.append(_rank_text(objs))
    blobs = []
    for o in objs[:5]:
        b = [f"({_form(o)})"]
        if "size" in keys:
            b.append(f"size={o['area_frac']:.3f}")
        if "position" in keys:
            b.append(f"at ({o['centroid_x']:.2f},{o['centroid_y']:.2f})")
        if "convexity" in keys:
            b.append(f"convexity={o['convexity']:.2f}")
        if "corners" in keys:
            b.append(f"corners={o['ncorners']}")
        if "angles" in keys:
            b.append(_angle_words(o))
        if "elongation" in keys:
            b.append(f"elongation={o['elongation']:.1f} long axis {_long_axis(o)}")
        if len(b) > 1:
            blobs.append(" ".join(b))
    body = "; ".join(f"figure{i+1} {s}" for i, s in enumerate(blobs))
    return " | ".join(head + ([body] if body else [])) or "no figures"

def ruler_words(feat, typical):
    """The ruler in plain words for the supervisor and the checker: size relative to the problem's typical figure,
    region, shape of the outline, corners. `typical` = median area of the largest figure over the 12 panels."""
    out = []
    for o in sorted(feat["objects"], key=lambda o: -o["area_frac"])[:3]:
        r = o["area_frac"] / max(typical, 1e-9)
        size = "small" if r < 0.5 else "large" if r > 2 else "medium"
        x, y = o["centroid_x"], o["centroid_y"]
        where = ("top" if y < 0.4 else "bottom" if y > 0.6 else "middle") + "-" + ("left" if x < 0.4 else "right" if x > 0.6 else "centre")
        axis = _long_axis(o)
        form = (f"elongated {axis if axis != 'none' else 'diagonal'}" if o["elongation"] > 1.8 else "compact")
        dents = "with dents" if o["convexity"] < 0.9 else "convex"
        out.append(f"{size} {form} {dents} {_form(o)} figure at {where}, {_angle_words(o)}")
    objs = sorted(feat["objects"], key=lambda o: -o["area_frac"])
    return "; ".join(out + ([_rank_text(objs)] if len(objs) > 1 else [])) or "empty"

def panel_records(bp, mode="hybrid", words=False):
    """Full per-panel record. mode: 'cv' (ruler only) | 'vlm' (eye only) | 'hybrid' (eye + ruler, divided).
    words=True gives the ruler in plain words (no numbers), for the supervisor and the checker."""
    cv = load_cv(bp)
    vf = vlm_fields(bp) if mode in ("vlm", "hybrid") else {}
    typical = float(np.median([max([o["area_frac"] for o in cv[p]["objects"]] or [0]) for p in PIDS]))
    recs = {}
    for pid in PIDS:
        eye = eye_text(vf.get(pid))
        ruler = ruler_words(cv[pid], typical) if words else ruler_text(cv[pid])
        recs[pid] = ruler if mode == "cv" else eye if mode == "vlm" else f"SEEN: {eye} || MEASURED: {ruler}"
    return recs

def records_block(recs):
    left = "\n".join(f"  L{k}: {recs[f'left_{k}']}" for k in range(1, 7))
    right = "\n".join(f"  R{k}: {recs[f'right_{k}']}" for k in range(1, 7))
    return left, right

def make_rule(left, right):
    return f"Left = {left.strip()}, Right = {right.strip()}" if left and left.strip() else ""

_NO_RULE = re.compile(r"\bno (consistent |clear |simple |single )?(rule|pattern)\b|\bunknown\b", re.I)
_EMPTY = {"none", "n/a", "na", "not applicable", "-", "nothing"}
def as_answer(out):
    """Normalise an agent's JSON answer to {has_rule, left, right, confidence, evidence, rule}."""
    left, right = str(out.get("left", "") or "").strip(), str(out.get("right", "") or "").strip()
    has = (bool(out.get("has_rule", True)) and bool(left) and not _NO_RULE.search(left)
           and left.lower().strip(" .") not in _EMPTY)
    try:
        conf = max(0.0, min(1.0, float(out.get("confidence", 0))))
    except Exception:
        conf = 0.0
    return {"has_rule": has, "left": left if has else "", "right": right if has else "",
            "confidence": conf if has else 0.0, "evidence": str(out.get("evidence", ""))[:300],
            "rule": make_rule(left, right) if has else ""}

## The grader — does a produced rule mean the same concept as Bongard's answer? (used by 04 and 05)

In [ ]:
# the pipeline's grader prompt (prompts/judge/semantic_equivalence.v1.txt in the original repo)
GRADER_PROMPT = """You are a strict grader for Bongard-problem solutions. Decide whether a PRODUCED
rule means the SAME CONCEPT as the reference concept. Judge MEANING, not wording.

REFERENCE CONCEPT (ground truth):
  concept   : {concept}
  left side : {left_rule}
  right side: {right_rule}

PRODUCED RULE (to grade):
  {produced_rule}

RULES FOR GRADING:
- "match" ONLY if the produced rule captures the same discriminating concept AND
  assigns it to the correct sides (left vs right). Synonyms/paraphrases are fine.
- A DIRECTION FLIP is a MISMATCH: if the reference says left=large/right=small and
  the produced rule says left=small/right=large, that is "mismatch" (not "partial").
- "partial" if it names the right dimension (e.g. size) but is vague, incomplete,
  or only half-correct.
- "mismatch" if it is about a different concept/dimension, or is empty/uninformative.
- Ignore surface wording, formatting, and extra commentary.

Respond with ONLY this JSON object:
{{"verdict": "match" | "partial" | "mismatch", "equivalent": true | false, "reason": "<one short sentence>"}}
Set "equivalent" true if and only if "verdict" is "match"."""

_GRADES = load_json("grades", "grades.json", default={})

def grade_once(bp, rule, model=None):
    gl, gr = split_rule(gt(bp))
    out = text_json(GRADER_PROMPT.format(concept=gt(bp), left_rule=gl, right_rule=gr, produced_rule=rule),
                    model=model or GRADER_MODEL, max_tokens=800, effort="low")
    return str(out.get("verdict", "")).lower() == "match" or out.get("equivalent") is True

def grade(bp, rule, votes=3):
    """Majority of `votes` grader calls. Cached on disk, so re-running costs nothing."""
    if not rule or not split_rule(rule)[0]:
        return False
    key = f"{GRADER_MODEL}||{bp}||{rule}"
    if key not in _GRADES:
        v = [grade_once(bp, rule) for _ in range(votes)]
        _GRADES[key] = {"votes": v, "match": sum(v) * 2 > len(v)}
        save_json(_GRADES, "grades", "grades.json")
    return _GRADES[key]["match"]

## Saving predictions

In [ ]:
def save_predictions(arm, rows):
    return save_json({"arm": arm, "rows": list(rows.values()) if isinstance(rows, dict) else rows},
                     "predictions", f"{arm}.json")

def load_predictions(arm):
    d = load_json("predictions", f"{arm}.json")
    return {r["bp"]: r for r in d["rows"]} if d else {}

def list_arms():
    return sorted(f[:-5] for f in os.listdir(os.path.join(RESULTS, "predictions")) if f.endswith(".json"))
print("00_common loaded.")